In [ ]:
import os
import sys
from pathlib import Path

project_root = None
for base in [Path.cwd(), Path.cwd().parent]:
    if (base / "src").exists():
        project_root = base
        break

if project_root is None:
    project_root = Path.cwd()

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

os.chdir(project_root)

from digital_twin import FleetDigitalTwin
from simulator import CMAPSSSimulator

model_path = project_root / "models" / "lstm_rul.pt"
scaler_path = project_root / "models" / "scaler.pkl"
config_path = project_root / "models" / "model_config.json"
data_path = project_root / "data" / "raw" / "train_FD001.txt"

# Initialize
fleet = FleetDigitalTwin(
    model_path=str(model_path),
    scaler_path=str(scaler_path),
    config_path=str(config_path),
)
simulator = CMAPSSSimulator(data_path=str(data_path))
print("Initialized successfully!")

In [ ]:
# Simulate 5 engines
sample_engines = simulator.get_sample_engines(n=5)

for engine_id in sample_engines:
    fleet.add_engine(engine_id)
    for sensor_data in simulator.stream_engine(engine_id):
        state = fleet.ingest(engine_id, sensor_data)

print("Simulation complete!")
print(f"\nFleet Summary:")
summary = fleet.get_fleet_summary()
for key, val in summary.items():
    print(f"   {key}: {val}")

In [ ]:
# Show fleet status
print("Fleet Status (sorted by RUL):\n")
for engine in fleet.get_fleet_status():
    print(f"Engine {engine['engine_id']:3d} | "
          f"Status: {engine['status']:12s} | "
          f"RUL: {str(engine['current_rul']):6s} cycles | "
          f"Health: {engine['health_score']}%")